In [2]:
!pip install pandas numpy scikit-learn


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

# Feature Engineering, Normalization, Standardization and PCA

# Step 1: Import Libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

# Step 2: Load Your CSV Dataset
df = pd.read_csv("students.csv")

print("Original Dataset:")
print(df.head())

# Step 3: Feature Engineering
# Select numerical columns
features = df.select_dtypes(include=[np.number]).copy()

# Handle missing values
features = features.fillna(features.mean())

# Create a new feature using the first two numerical columns
if features.shape[1] >= 2:
    features['New_Feature'] = (
        features.iloc[:, 0] * features.iloc[:, 1]
    )

print("\nAfter Feature Engineering:")
print(features.head())

# Step 4: Min-Max Normalization (0 to 1)
minmax = MinMaxScaler()

normalized = minmax.fit_transform(features)

normalized_df = pd.DataFrame(
    normalized,
    columns=features.columns
)

print("\nMin-Max Normalization:")
print(normalized_df.head())

# Step 5: Z-Score Standardization (Mean=0, Std=1)
scaler = StandardScaler()

standardized = scaler.fit_transform(features)

standardized_df = pd.DataFrame(
    standardized,
    columns=features.columns
)

print("\nZ-Score Standardization:")
print(standardized_df.head())

# Step 6: PCA - Dimensionality Reduction
if standardized.shape[1] >= 2:
    pca = PCA(n_components=2)

    principal_components = pca.fit_transform(standardized)

    pca_df = pd.DataFrame(
        principal_components,
        columns=['PC1', 'PC2']
    )

    print("\nPCA - Principal Components:")
    print(pca_df.head())

    # Step 7: Explained Variance
    print("\nExplained Variance Ratio:")
    print(pca.explained_variance_ratio_)

    print("\nTotal Explained Variance:")
    print(sum(pca.explained_variance_ratio_))

else:
    print("\nPCA requires at least 2 numerical features.")

# Step 8: Final Output
print("\nProgram Executed Successfully!")


Original Dataset:
  StudentID  StudyHours  Attendance  PrevMarks  Marks Branch Result
0     S1000        5.08       82.32      65.44  82.46    ISE   Pass
1     S1001        6.16       68.27      82.55  79.14    CSD   Pass
2     S1002        5.42       98.16      67.71  76.89    CSE   Pass
3     S1003        2.93       79.50      68.25  62.16    ISE   Fail
4     S1004        8.01       93.33      60.55  86.49    ISE   Pass

After Feature Engineering:
   StudyHours  Attendance  PrevMarks  Marks  New_Feature
0        5.08       82.32      65.44  82.46     418.1856
1        6.16       68.27      82.55  79.14     420.5432
2        5.42       98.16      67.71  76.89     532.0272
3        2.93       79.50      68.25  62.16     232.9350
4        8.01       93.33      60.55  86.49     747.5733

Min-Max Normalization:
   StudyHours  Attendance  PrevMarks     Marks  New_Feature
0    0.349415    0.690151   0.508812  0.672883     0.377907
1    0.428363    0.443919   0.751990  0.610966     0.380183


In [ ]:

# ============================================
# STUDENT PERFORMANCE DATA ANALYSIS
# Feature Engineering, Data Cleaning,
# Missing Values, Outliers, Z-Score,
# Normalization, Standardization and PCA
# ============================================

# STEP 1: IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    LabelEncoder
)

from sklearn.decomposition import PCA



# STEP 2: LOAD DATASET

from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select Student CSV File",
    filetypes=[("CSV Files", "*.csv")]
)

df = pd.read_csv(file_path)

print("ORIGINAL DATASET:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
print(df.info())


# STEP 3: CHECK MISSING VALUES

print("\nMISSING VALUES:")
print(df.isnull().sum())

# Fill missing numerical values with column mean
numeric_columns = df.select_dtypes(include=np.number).columns

df[numeric_columns] = df[numeric_columns].fillna(
    df[numeric_columns].mean()
)

# Fill missing categorical values with mode
categorical_columns = df.select_dtypes(
    include=['object']
).columns

for col in categorical_columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nAfter Handling Missing Values:")
print(df.isnull().sum())


# STEP 4: REMOVE DUPLICATE RECORDS

print("\nDuplicate Records:")
print(df.duplicated().sum())

df = df.drop_duplicates()

print("\nAfter Removing Duplicates:")
print(df.shape)


# STEP 5: OUTLIER DETECTION USING IQR

print("\nOUTLIER DETECTION:")

numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) | (df[col] > upper)
    ]

    print("\nColumn:", col)
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print("Number of Outliers:", len(outliers))


# STEP 6: Z-SCORE CALCULATION

print("\nZ-SCORE CALCULATION:")

z_scores = (
    df[numeric_columns] -
    df[numeric_columns].mean()
) / df[numeric_columns].std(ddof=0)

print(z_scores)

# Detect possible outliers using Z-score
z_outliers = (z_scores.abs() > 3)

print("\nZ-Score Outliers:")
print(z_outliers.sum())


# STEP 7: DATA CLEANING

# Remove rows having extreme Z-score values
# Threshold is 3 standard deviations

clean_df = df[
    (z_scores.abs() <= 3).all(axis=1)
].copy()

print("\nDataset After Outlier Cleaning:")
print(clean_df)

print("\nCleaned Dataset Shape:")
print(clean_df.shape)


# STEP 8: FEATURE ENGINEERING

# Create a new feature using study hours and attendance
clean_df['Study_Attendance_Score'] = (
    clean_df['StudyHours'] * clean_df['Attendance'] / 100
)

# Create total performance score
clean_df['Performance_Score'] = (
    clean_df['PrevMarks'] + clean_df['Marks']
) / 2

print("\nAfter Feature Engineering:")
print(clean_df.head())


# STEP 9: ENCODING CATEGORICAL DATA

# Convert Branch into numerical values
le = LabelEncoder()

clean_df['Branch_Encoded'] = le.fit_transform(
    clean_df['Branch']
)

# Convert Result into numerical values
clean_df['Result_Encoded'] = clean_df['Result'].map(
    {'Pass': 1, 'Fail': 0}
)

print("\nAfter Encoding:")
print(clean_df.head())


# STEP 10: SELECT FEATURES FOR SCALING

features = clean_df[
    [
        'StudyHours',
        'Attendance',
        'PrevMarks',
        'Marks',
        'Study_Attendance_Score',
        'Performance_Score'
    ]
]


# STEP 11: MIN-MAX NORMALIZATION

minmax = MinMaxScaler()

normalized = minmax.fit_transform(features)

normalized_df = pd.DataFrame(
    normalized,
    columns=features.columns,
    index=features.index
)

print("\nMIN-MAX NORMALIZATION (0 TO 1):")
print(normalized_df)


# STEP 12: Z-SCORE STANDARDIZATION

scaler = StandardScaler()

standardized = scaler.fit_transform(features)

standardized_df = pd.DataFrame(
    standardized,
    columns=features.columns,
    index=features.index
)

print("\nZ-SCORE STANDARDIZATION:")
print(standardized_df)


# STEP 13: PCA - DIMENSIONALITY REDUCTION

pca = PCA(n_components=2)

principal_components = pca.fit_transform(standardized)

pca_df = pd.DataFrame(
    principal_components,
    columns=['PC1', 'PC2'],
    index=features.index
)

print("\nPCA - PRINCIPAL COMPONENTS:")
print(pca_df)


# STEP 14: EXPLAINED VARIANCE

print("\nEXPLAINED VARIANCE RATIO:")
print(pca.explained_variance_ratio_)

print("\nTOTAL EXPLAINED VARIANCE:")
print(sum(pca.explained_variance_ratio_))


# STEP 15: DISPLAY FINAL RESULTS

print("\nFINAL CLEANED DATASET:")
print(clean_df.head())

print("\nPROGRAM EXECUTED SUCCESSFULLY!")


# STEP 16: SAVE OUTPUT FILES

clean_df.to_csv("cleaned_student_data.csv", index=False)

normalized_df.to_csv("normalized_student_data.csv", index=False)

standardized_df.to_csv("standardized_student_data.csv", index=False)

pca_df.to_csv("pca_student_data.csv", index=False)

print("\nAll output CSV files saved successfully!")


In [ ]:

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

# STEP 1: Enter Student Dataset Directly

data = {
    'StudentID': ['S1000','S1001','S1002','S1003','S1004',
                  'S1005','S1006','S1007','S1008','S1009',
                  'S1010','S1011','S1012','S1013','S1014'],

    'StudyHours': [5.08,6.16,5.42,2.93,8.01,7.15,7.24,3.5,
                   8.59,2.42,6.79,10.32,3.32,4.25,5.72],

    'Attendance': [82.32,68.27,98.16,79.5,93.33,78.88,100,
                   100,74.76,90.63,86.39,95.79,65.46,86.92,91.76],

    'PrevMarks': [65.44,82.55,67.71,68.25,60.55,65.15,70.62,
                  40.35,45.78,77.15,68.56,63.24,66.28,71.21,57.9],

    'Marks': [82.46,79.14,76.89,62.16,86.49,83.57,91.56,
              63.71,81.96,78.22,91.43,100,62.04,73.45,80.52],

    'Branch': ['ISE','CSD','CSE','ISE','ISE','CSD','CSE',
               'CSE','ISE','AIML','ISE','ISE','ISE','ISE','CSD'],

    'Result': ['Pass','Pass','Pass','Fail','Pass','Pass','Pass',
               'Fail','Pass','Pass','Pass','Pass','Fail','Pass','Pass']
}

df = pd.DataFrame(data)

print("ORIGINAL DATASET")
print(df)

# STEP 2: CHECK MISSING VALUES

print("\nMISSING VALUES")
print(df.isnull().sum())

# STEP 3: REMOVE DUPLICATES

df = df.drop_duplicates()

print("\nDUPLICATES REMOVED")
print(df.shape)

# STEP 4: OUTLIER DETECTION USING IQR

numeric_columns = ['StudyHours', 'Attendance', 'PrevMarks', 'Marks']

print("\nOUTLIER DETECTION")

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print("\nColumn:", col)
    print("Outliers:", len(outliers))
    print(outliers[col].tolist())

# STEP 5: Z-SCORE CALCULATION

z_scores = (
    df[numeric_columns] - df[numeric_columns].mean()
) / df[numeric_columns].std(ddof=0)

print("\nZ-SCORE VALUES")
print(z_scores)

print("\nZ-SCORE OUTLIERS")
print((z_scores.abs() > 3).sum())

# STEP 6: HANDLE MISSING VALUES

df[numeric_columns] = df[numeric_columns].fillna(
    df[numeric_columns].mean()
)

# STEP 7: FEATURE ENGINEERING

df['Study_Attendance_Score'] = (
    df['StudyHours'] * df['Attendance'] / 100
)

df['Performance_Score'] = (
    df['PrevMarks'] + df['Marks']
) / 2

print("\nFEATURE ENGINEERING")
print(df.head())

# STEP 8: MIN-MAX NORMALIZATION

features = df[
    ['StudyHours', 'Attendance', 'PrevMarks', 'Marks',
     'Study_Attendance_Score', 'Performance_Score']
]

minmax = MinMaxScaler()

normalized = minmax.fit_transform(features)

normalized_df = pd.DataFrame(
    normalized,
    columns=features.columns
)

print("\nMIN-MAX NORMALIZATION (0 TO 1)")
print(normalized_df)

# STEP 9: Z-SCORE STANDARDIZATION

scaler = StandardScaler()

standardized = scaler.fit_transform(features)

standardized_df = pd.DataFrame(
    standardized,
    columns=features.columns
)

print("\nSTANDARDIZATION (MEAN 0, STD 1)")
print(standardized_df)

# STEP 10: PCA DIMENSIONALITY REDUCTION

pca = PCA(n_components=2)

principal_components = pca.fit_transform(standardized)

pca_df = pd.DataFrame(
    principal_components,
    columns=['PC1', 'PC2']
)

print("\nPCA PRINCIPAL COMPONENTS")
print(pca_df)

# STEP 11: EXPLAINED VARIANCE

print("\nEXPLAINED VARIANCE RATIO")
print(pca.explained_variance_ratio_)

print("\nTOTAL EXPLAINED VARIANCE")
print(sum(pca.explained_variance_ratio_))

# STEP 12: SAVE OUTPUT FILES

df.to_csv("cleaned_student_data.csv", index=False)
normalized_df.to_csv("normalized_student_data.csv", index=False)
standardized_df.to_csv("standardized_student_data.csv", index=False)
pca_df.to_csv("pca_student_data.csv", index=False)

print("\nPROGRAM EXECUTED SUCCESSFULLY!")


In [ ]:
print("HELLO WORLD")